# IMPORT

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model
from peft.peft_model import PeftModel
from datasets import load_dataset
from torch.utils.data import Dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# DEVICE AND DATATYPE


In [ ]:
skip_train = False
device = torch.device('cuda:0') 
torch_dtype = torch.bfloat16

# LOAD MODEL

In [ ]:
MODEL_1 = "cardiffnlp/twitter-xlm-roberta-large-2022"
MODEL_2 = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
MODEL_3 = "cardiffnlp/twitter-xlm-roberta-base"
MODEL_4 = "deepset/gbert-base"
MODEL_5 = "Hate-speech-CNERG/dehatebert-mono-german"
MODEL_6 = "deepset/bert-base-german-cased-hatespeech-GermEval18Coarse"
MODEL_7 = "google-bert/bert-base-german-cased"
model_id = MODEL_3 #CONFIG
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=3, torch_dtype=torch_dtype)
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=True,
    torch_dtype=torch_dtype,
)

# LOAD DATASET

In [ ]:
trainset = load_dataset("csv", data_files="../data/train_classifier.tsv", delimiter="\t")["train"] if not skip_train else None
devset= load_dataset("csv", data_files="../data/test.tsv", delimiter="\t")["train"] if not skip_train else None
testset = load_dataset("csv", data_files="../data/processed_ger_test.tsv", delimiter="\t")["train"] 

print(trainset)
print(devset)
print(testset)

In [ ]:
# from datasets import concatenate_datasets

# # Merge devset and trainset
# trainset = trainset.rename_column("label", "toxic")
# merged_dataset = concatenate_datasets([devset, trainset])

# # Shuffle the merged dataset
# shuffled_dataset = merged_dataset.shuffle(seed=42)

# # Split into 90% training and 10% validation
# split_datasets = shuffled_dataset.train_test_split(test_size=0.1, seed=42)
# new_trainset = split_datasets['train']
# new_devset = split_datasets['test']

# print(new_trainset)
# print(new_devset)

In [ ]:
class TokenizedDataset(Dataset): #CONFIG
    def __init__(self, dataset, tokenizer):
        self.dataset = dataset
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        text = example["text"]
        if example["id"].startswith("ger"):
            label = 0
        elif example["id"].startswith("eng"):
            label = 1
        elif example["id"].startswith("fin"):
            label = 2
        else :
            raise ValueError("Unknown label")
        
            
        tokenized = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokenized["input_ids"].squeeze(0),
            "attention_mask": tokenized["attention_mask"].squeeze(0),
            "label": label
        }

In [ ]:
tokenized_devset = TokenizedDataset(devset, tokenizer) if not skip_train else None
tokenized_trainset = TokenizedDataset(trainset, tokenizer) if not skip_train else None
tokenized_testset = TokenizedDataset(testset, tokenizer)

# LoRA

In [ ]:
lora_config = LoraConfig( #CONFIG
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False,
    r=8,
    lora_alpha=64,
    lora_dropout=0.2,
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

# TRAINER

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(pred):
    logits, labels = pred.predictions, pred.label_ids
    preds = logits.argmax(axis=-1)  # Pick class with highest probability

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [ ]:
training_args = TrainingArguments( #CONFIG
    output_dir=f"./results",
    eval_strategy="epoch",
    learning_rate=1e-5,  # Reduced learning rate
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=10,  
    weight_decay=1e-3,  
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch"
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_trainset,
    eval_dataset=tokenized_devset,
    compute_metrics=compute_metrics
)

In [ ]:
if not skip_train:
    trainer.train()

In [ ]:
def compute_metrics(pred):
    threshold=0.5
    logits, labels = pred.predictions, pred.label_ids
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = (probs[:, 1] > threshold).astype(int)  # Apply threshold

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

checkpoint_dir = "combined-dset-google-epoch-9-is-best/checkpoint-19575"  #CONFIG

peft_model = PeftModel.from_pretrained(model, model_id=checkpoint_dir)
trainer.model = peft_model
trainer.args.per_device_eval_batch_size = 16
trainer.compute_metrics = compute_metrics

evaluation_results = trainer.evaluate()
print(evaluation_results)

# INFERENCE


In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import torch.nn.functional as F
import torch

peft_model = PeftModel.from_pretrained(model, checkpoint_dir)
trainer.model = peft_model
trainer.compute_metrics = None

inferenceset = load_dataset("csv", data_files="../data/processed_ger_test.tsv", delimiter="\t")["train"]
tokenized_inferenceset = TokenizedDataset(inferenceset, tokenizer)
predictions_output = trainer.predict(tokenized_inferenceset)

logits = predictions_output.predictions
probs = F.softmax(torch.tensor(logits), dim=1).numpy()

threshold = 0.5
predicted_labels = (probs[:, 1] >= threshold).astype(int)

original_data = pd.read_csv("../data/processed_ger_test.tsv", sep="\t")

df_submission = pd.DataFrame({
    "id": original_data["id"],
    "predicted": predicted_labels
})

df_submission.to_csv("submission_ger.tsv", sep="\t", index=False)
print("Submission file saved as submission_ger.tsv")
